# Moral foundation coding with an OpenAI model — main analysis

Primary cell: **main prompt × 0-shot**, model `gpt-6-luna` (non-reasoning, `reasoning_effort="none"`).

Before running:
* Put `run_grid_openai.py` in the same folder as this notebook.
* Select the Python kernel (top right) you want to use.
* Provide your OpenAI API key (prompted below, or set `OPENAI_API_KEY` in a `.env` file — keep `.env` out of git).

## Setup

In [ ]:
# Run once, then restart the kernel
%pip install -q -U openai pandas tqdm scikit-learn ipywidgets python-dotenv

In [ ]:
# Reload run_grid_openai.py automatically whenever you edit it
%load_ext autoreload
%autoreload 2

In [ ]:
import os, json, getpass
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from .env if present
except ImportError:
    pass
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

import run_grid_openai as rg
print("model:", rg.MODEL, "| reasoning_effort:", rg.REASONING_EFFORT, "| output folder:", rg.OUT_DIR)

## Load validation set

In [ ]:
validation = rg.load_validation()
print(validation.shape)
validation.head()

## Prompt
The main prompt is identical to the one in `llm_moral_coding.ipynb`.

In [ ]:
print(rg.make_prompt(validation["text"][0]))

Test on one paragraph:

In [ ]:
rg.check_params(rg.make_prompt(validation["text"][0]))   # drops temperature/seed if the model rejects them
rg.ask_llm(rg.make_prompt(validation["text"][0]), validation["text"][0])

## Run the primary cell (main × 0-shot)

Every paragraph is coded once; no resume logic. Results are saved to
`runs/gpt-6-luna/main_0shot_validation.csv`, together with the exact prompt (`_prompt.txt`)
and the settings used (`_settings.json`).

In [ ]:
pred = rg.run_cell(validation, "main", 0)
pred.head()

In [ ]:
settings = json.load(open(f"{rg.OUT_DIR}/main_0shot_validation_settings.json"))
print("dropped parameters:", settings["dropped_params"])     # [] = temperature and seed were used
print("model versions returned:", settings["model_versions"])
print("params:", settings["params"])

## Output checks

In [ ]:
print("problems:")
print(pred["problem"].fillna("none").value_counts())
warnings = pred["warnings"].apply(lambda w: w if isinstance(w, list) else [])
print("\nwarnings:")
print(pd.Series([x.split(": ", 1)[1] for ws in warnings for x in ws], dtype=str).value_counts())

## Evaluation against human labels

In [ ]:
EVAL_LABELS = ["moral"] + rg.FOUNDATIONS
CODERS = ["Yunfan", "Wiktoria", "Carlos"]
LLM_COL = {"moral": "moral_rhetoric", **{f: f for f in rg.FOUNDATIONS}}

m = validation.merge(pred.drop(columns=["text"], errors="ignore"), on="para_id")
m = m[m["moral_rhetoric"].notna()]          # drop parse/API errors
for l in EVAL_LABELS:                        # majority vote of the three coders
    m[f"{l}_majority"] = (m[[f"{l}_{c}" for c in CODERS]].sum(axis=1) >= 2).astype(int)

def evaluate(df, human_col_fn, llm_col_map=LLM_COL):
    rows = []
    for l in EVAL_LABELS:
        sub = df[[human_col_fn(l), llm_col_map[l]]].dropna()
        y, yhat = sub.iloc[:, 0].astype(int), sub.iloc[:, 1].astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, yhat, average="binary", zero_division=0)
        rows.append({"label": l, "n": len(sub), "human_n": y.sum(), "llm_n": yhat.sum(),
                     "accuracy": (y == yhat).mean(), "precision": p, "recall": r, "f1": f,
                     "kappa": cohen_kappa_score(y, yhat)})
    return pd.DataFrame(rows).set_index("label").round(3)

all_results = {c: evaluate(m, lambda l, c=c: f"{l}_{c}") for c in CODERS}
all_results["majority"] = evaluate(m, lambda l: f"{l}_majority")
results = pd.concat(all_results, names=["coder", "label"])

print("n =", len(m))
for c, res in all_results.items():
    print(f"{c:>9} macro-F1 (5 foundations):", res.loc[rg.FOUNDATIONS, "f1"].mean().round(3))
results

In [ ]:
results.to_csv(f"{rg.OUT_DIR}/main_0shot_evaluation.csv")

### LLM–human agreement vs. human–human agreement (Cohen's κ)

Is the LLM within the range of agreement among the human coders?

In [ ]:
from itertools import combinations

rows = []
for l in EVAL_LABELS:
    hh = [cohen_kappa_score(m[f"{l}_{a}"].astype(int), m[f"{l}_{b}"].astype(int))
          for a, b in combinations(CODERS, 2)]
    lh = [cohen_kappa_score(m[f"{l}_{c}"].astype(int), m[LLM_COL[l]].astype(int)) for c in CODERS]
    rows.append({"label": l,
                 "human-human mean": sum(hh) / 3, "human-human min": min(hh), "human-human max": max(hh),
                 "LLM-human mean": sum(lh) / 3, "LLM-human min": min(lh), "LLM-human max": max(lh)})
pd.DataFrame(rows).set_index("label").round(3)

## Disagreements for error analysis

In [ ]:
EVID_COL = {l: f"evidence_{LLM_COL[l]}" for l in EVAL_LABELS}

def find_disagreements(df, coder="majority", outfile=None):
    """coder: 'Yunfan' / 'Wiktoria' / 'Carlos' / 'majority'"""
    def diff(row):
        out = []
        for l in EVAL_LABELS:
            h, p = row[f"{l}_{coder}"], row[LLM_COL[l]]
            if pd.isna(h) or pd.isna(p):
                continue
            if int(h) != int(p):
                out.append(l)
        return out

    df = df.copy()
    df["disagree_on"] = df.apply(diff, axis=1)
    errors = df[df["disagree_on"].str.len() > 0]
    cols = (["para_id", "text", "disagree_on"]
            + [c for l in EVAL_LABELS for c in (f"{l}_{coder}", LLM_COL[l])]
            + [EVID_COL[l] for l in EVAL_LABELS])
    if outfile:
        errors[cols].to_csv(outfile, index=False)
    print(f"{coder}: {len(errors)} / {len(df)} paragraphs with at least one disagreement")
    return errors[cols]

errors = find_disagreements(m, coder="majority",
                            outfile=f"{rg.OUT_DIR}/main_0shot_disagreements_majority.csv")
errors

## Robustness cells (run later)

Before running these:
* **paraphrase** — reword `DEFINITIONS_PARAPHRASE` in `run_grid_openai.py` (now a copy of the main definitions).
* **2- / 5-shot** — create `fewshot_examples.csv` (columns: `para_id, text`, the six labels with 1 = present, optional `evidence_<label>`). Example paragraphs that are also in the validation set are left out of that cell.
* **reverse** — nothing to prepare; codes are flipped back to 1 = present when saved.

In [ ]:
# all_pred = rg.run_grid(validation)                                 # all 9 cells
# all_pred = rg.run_grid(validation, cells=[("reverse", 0)])         # a single robustness cell